In [ ]:
# Run this cell to install required dependencies for Google Colab GPU training
!pip install -q transformers datasets peft accelerate sentencepiece evaluate sacrebleu bitsandbytes==0.48.0 "torchao>=0.16.0" "torch>=2.11.0" torchvision torchaudio > /dev/null 2>&1


In [ ]:
# Remove the default sample_data directory created by Google Colab
!rm -rf /content/sample_data


# Machine Translation: English to Kamba (and Kamba to English)

This notebook trains an NMT model using `google/mt5-base` to translate between English and Kamba using the Hugging Face `transformers` library.

In [1]:
import os
os.environ['BNB_CUDA_VERSION'] = '121'
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore", module="torchao.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes.*")
warnings.filterwarnings("ignore", module="torch.utils._pytree")
import logging
logging.getLogger("torch.utils._pytree").setLevel(logging.ERROR)
import logging
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

# Load the dataset
dataset = load_dataset('michsethowusu/english-kamba_sentence-pairs_mt560')

# Display the structure and a few examples
print(dataset)
if 'train' in dataset:
    print("\nSample:", dataset['train'][0])

    # Colab Pro Optimization: Split dataset into 95% train, 5% validation
    if "validation" not in dataset:
        split_dataset = dataset['train'].train_test_split(test_size=0.05, seed=42)
        dataset['train'] = split_dataset['train']
        dataset['validation'] = split_dataset['test']
        print("\nAfter split:", dataset)
else:
    print("\nSample:", dataset[0])
    # Colab Pro Optimization: Split dataset into 95% train, 5% validation
    if "validation" not in dataset:
        dataset = dataset.train_test_split(test_size=0.05, seed=42)
        dataset["validation"] = dataset.pop("test")
        print("\nAfter split:", dataset)


DatasetDict({
    train: Dataset({
        features: ['eng', 'kam'],
        num_rows: 51054
    })
})

Sample: {'eng': '" Happy are the mild - tempered , since they will inherit the earth . " - Matt .', 'kam': '" Nĩ aathime ala auu : nũndũ nĩo makatiĩwa nthĩ . " - Mt .'}


## 2. Preprocessing
We need to tokenize the inputs and targets using `AutoTokenizer`. For mT5, we prefix tasks to guide translation.

In [2]:
from transformers import AutoTokenizer
import multiprocessing
import warnings
warnings.filterwarnings("ignore", module="torchao.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes.*")
warnings.filterwarnings("ignore", module="torch.utils._pytree")
import logging
logging.getLogger("torch.utils._pytree").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*byte fallback option.*")

model_checkpoint = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, legacy=False)

max_input_length = 256
max_target_length = 256

def preprocess_function(examples):
    keys = list(examples.keys())
    
    if "translation" in keys:
        en_texts = [ex["en"] for ex in examples["translation"]]
        kam_texts = [ex["kam"] for ex in examples["translation"]]
    else:
        en_col = next((c for c in ["english", "en", "English", "source"] if c in keys), keys[0])
        kam_col = next((c for c in ["kamba", "kam", "Kamba", "target"] if c in keys), keys[1] if len(keys)>1 else keys[0])
        en_texts = [str(ex) for ex in examples[en_col]]
        kam_texts = [str(ex) for ex in examples[kam_col]]
        
    inputs = []
    targets = []
    
    # 1. English to Kamba
    inputs.extend(["translate English to Kamba: " + text for text in en_texts])
    targets.extend(kam_texts)
    
    # 2. Kamba to English (Bidirectional Training!)
    inputs.extend(["translate Kamba to English: " + text for text in kam_texts])
    targets.extend(en_texts)
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True, padding="max_length")
        
    labels_with_ignore_index = []
    for label_sequence in labels["input_ids"]:
        labels_with_ignore_index.append(
            [l if l != tokenizer.pad_token_id else -100 for l in label_sequence]
        )
    model_inputs["labels"] = labels_with_ignore_index
    return model_inputs

num_cores = multiprocessing.cpu_count()
print(f"Using {num_cores} cores for tokenization...")

# Remove original columns since we are doubling the number of rows (bidirectional)
column_names = dataset["train"].column_names if "train" in dataset else dataset.column_names

tokenized_datasets = dataset.map(
    preprocess_function, 
    batched=True, 
    num_proc=num_cores,
    remove_columns=column_names
)


Using 16 cores for tokenization...


## 3. Model Setup & Training

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import multiprocessing
try:
    multiprocessing.set_start_method("fork", force=True)
except RuntimeError:
    pass
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

import warnings
warnings.filterwarnings("ignore", module="torchao.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes.*")
warnings.filterwarnings("ignore", module="torch.utils._pytree")
import logging
logging.getLogger("torch.utils._pytree").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Can\'t initialize NVML")

# Check if GPU is actually available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- Initializing Model ---")
print(f"Hardware Device: {device.upper()}")

# Load model and suppress the word embeddings warning
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, quantization_config=quantization_config, tie_word_embeddings=False)
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
from peft import get_peft_model, LoraConfig, TaskType
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=64, 
    lora_alpha=32, 
    lora_dropout=0.05, 
    bias="none", 
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"], 
    modules_to_save=["shared"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

batch_size = 8 # Reduced from 32 to prevent OOM on mt5-base
import evaluate
import numpy as np

sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    bleu_result = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels, force=True)
    chrf_result = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": bleu_result["score"], "chrf": chrf_result["score"]}

args = Seq2SeqTrainingArguments(
    "mt5-english-kamba",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=32, # Increased from 8 to maintain effective batch size 256
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=256,
    use_cpu=False,
    fp16=False,
    bf16=True, # L4 GPUs support bf16 for faster, more stable training
    # Speed Optimization 2: Dataloader Multiprocessing
    dataloader_num_workers=8, # High-RAM Colab provides more CPU cores
    dataloader_prefetch_factor=4,
    # Speed Optimization 3: Only pin memory if on GPU to avoid PyTorch warnings
    dataloader_pin_memory=(device == "cuda"),
)

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

--- Initializing Model ---
Hardware Device: CPU


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


In [ ]:
# Save the trained model and tokenizer as a precaution
save_directory = "./mt5-english-kamba-final"
print(f"Saving model and tokenizer to {save_directory}...")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print("Model and tokenizer successfully saved. Continuing with the model in memory for inference!")

## 4. Inference
Here we define our translation functions.

In [ ]:
import torch

def translate_en_to_kam(text):
    model.eval()
    input_text = "translate English to Kamba: " + text
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, max_length=128, num_beams=4, early_stopping=True)
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def translate_kam_to_en(text):
    model.eval()
    input_text = "translate Kamba to English: " + text
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, max_length=128, num_beams=4, early_stopping=True)
        
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_sentence = "translate English to Kamba: How are you doing today?"
inputs = tokenizer(test_sentence, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_length=50)

# Set skip_special_tokens to FALSE to see the raw tokens
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("Raw Output:", raw_output)

### Interactive Translation (English -> Kamba)
Run the cell below to type your own English sentences and translate them to Kamba.

In [ ]:
user_input = input("Enter an English sentence to translate to Kamba: ")

if user_input.strip():
    translation = translate_en_to_kam(user_input)
    print("\n--- Results ---")
    print(f"English: {user_input}")
    print(f"Kamba:   {translation}")
else:
    print("No text entered!")

### Interactive Translation (Kamba -> English)
Run the cell below to type your own Kamba sentences and translate them to English.

In [ ]:
user_input = input("Enter a Kamba sentence to translate to English: ")

if user_input.strip():
    translation = translate_kam_to_en(user_input)
    print("\n--- Results ---")
    print(f"Kamba:   {user_input}")
    print(f"English: {translation}")
else:
    print("No text entered!")

In [ ]:
import evaluate
import numpy as np
from tqdm.auto import tqdm

print("Loading evaluation metrics...")
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# Use a subset of validation for holistic evaluation to save time (e.g., 200 sentences)
eval_subset = dataset['validation'].select(range(min(200, len(dataset['validation']))))
references = []
predictions = []

print(f"Evaluating on {len(eval_subset)} samples...")
model.eval()
for ex in tqdm(eval_subset):
    # Handle schema
    keys = list(ex.keys())
    if 'translation' in keys:
        en_text = ex['translation']['en']
        kam_text = ex['translation']['kam']
    else:
        en_col = next((c for c in ['english', 'en', 'English', 'source'] if c in keys), keys[0])
        kam_col = next((c for c in ['kamba', 'kam', 'Kamba', 'target'] if c in keys), keys[1])
        en_text = str(ex[en_col])
        kam_text = str(ex[kam_col])
    
    pred = translate_en_to_kam(en_text)
    predictions.append(pred)
    references.append([kam_text])

bleu_results = sacrebleu.compute(predictions=predictions, references=references, force=True)
chrf_results = chrf.compute(predictions=predictions, references=references)

print("\n--- Final Holistic Evaluation ---")
print(f"BLEU Score: {bleu_results['score']:.2f}")
print(f"chrF Score: {chrf_results['score']:.2f}")
